# 03 - Preparação do Modelo (Data Preparation)

Este notebook foca na preparação dos dados para a fase de modelagem, seguindo o framework CRISP-DM. Aqui realizaremos o tratamento de variáveis categóricas, escalonamento de atributos numéricos e a divisão dos dados para treinamento e avaliação.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# Configurações de visualização
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados

Carregamos o dataset processado no EDA.

In [2]:
# Carregar o dataset
data_path = os.path.join('..', 'data', 'processed', 'synthetic_energy_emissions_dataset.csv')
df = pd.read_csv(data_path)

# Derivar colunas de mes e season a partir da coluna data
df['data'] = pd.to_datetime(df['data'])
df['mes'] = df['data'].dt.month

def get_season(mes):
    if mes in [12, 1, 2]: return 'Verao'
    if mes in [3, 4, 5]: return 'Outono'
    if mes in [6, 7, 8]: return 'Inverno'
    return 'Primavera'

df['season'] = df['mes'].apply(get_season)

print(f'Formato do dataset: {df.shape}')
df.head()

Formato do dataset: (100000, 15)


,Unnamed: 0,id_empresa,data,estado,setor,porte,tipo_combustivel,consumo_kwh,fonte_energia,emissao_co2,modal_transporte,km_transporte,emissao_transporte_co2,mes,season
0,0,C228154,2025-09-02,DF,industrial,small,electric,19691.786482,hidrelétrica,85.154433,moto,820.53,92.7198,9,Primavera
1,1,C544863,2025-07-06,MG,outros,large,electric,5884.049610,eólica,61.693557,metro_trem,186.66,1.1200,7,Inverno
2,2,C898300,2025-04-13,SP,residencial,small,electric,303.184946,hidrelétrica,1.083152,carro,111.98,21.5010,4,Outono
3,3,C980827,2025-06-13,MS,residencial,small,electric,98.789450,hidrelétrica,0.377149,carro,291.33,55.9352,6,Inverno
4,4,C319070,2025-08-22,BA,residencial,small,electric,177.441915,eólica,2.128812,metro_trem,518.29,3.1097,8,Inverno


## 2. Seleção de Atributos (Feature Selection)

Definimos nossas variáveis independentes (X) e a variável alvo (y - `co2_emission`).

In [3]:
target = 'emissao_co2'
features = ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'season']

X = df[features]
y = df[target]

print(f"Variavel alvo: {target}")
print(f"Atributos ({len(features)}): {features}")

Variavel alvo: emissao_co2
Atributos (6): ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'season']


## 3. Divisão Treino e Teste

Dividimos os dados em 80% para treinamento e 20% para teste.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho Treino: {X_train.shape}")
print(f"Tamanho Teste: {X_test.shape}")

Tamanho Treino: (80000, 6)
Tamanho Teste: (20000, 6)


## 4. Pré-processamento (Pipeline)

Aplicamos as transformações necessárias de forma consistente.

In [5]:
# Identificar colunas numericas e categoricas
numeric_features = ['consumo_kwh', 'mes']
categorical_features = ['setor', 'estado', 'fonte_energia', 'season']  # era 'estacao'

# Definir transformadores
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('[OK] Pipeline de pre-processamento configurado.')


[OK] Pipeline de pre-processamento configurado.


## 5. Conclusão — Preparação dos Dados

Este notebook concluiu a fase de **Data Preparation** do ciclo CRISP-DM, entregando um pipeline de pré-processamento robusto e reproduzível.

### O que foi construído

| Etapa | Decisão | Justificativa |
|---|---|---|
| Features selecionadas | `consumo_kwh`, `fonte_energia`, `estado`, `setor`, `mes`, `season` | Validadas pela análise exploratória do notebook 02 |
| Normalização numérica | `StandardScaler` em `consumo_kwh` e `mes` | Evita que a escala de kWh domine o aprendizado |
| Codificação categórica | `OneHotEncoder(handle_unknown='ignore')` | Permite inferência em estados/setores não vistos no treino |
| Divisão treino/teste | 80% / 20% com `random_state=42` | Garante reprodutibilidade e volume suficiente para avaliação |
| Encapsulamento | `sklearn.Pipeline` | Garante que o pré-processamento seja aplicado de forma idêntica em treino, teste e produção — eliminando data leakage |

### Por que o Pipeline do scikit-learn importa

Encapsular o `ColumnTransformer` dentro de um `Pipeline` é uma decisão de engenharia crítica: o `StandardScaler` aprende a média e desvio padrão **somente nos dados de treino**, e aplica essa mesma transformação no teste e em produção. Sem isso, haveria **data leakage** — o modelo 'veria' informações do futuro durante o treinamento, inflando artificialmente as métricas.

### Impacto de negócio

O pipeline garante que o modelo final exportado em `.joblib` (notebook 04) seja autocontido: ao receber novos dados de uma empresa, ele aplica automaticamente todas as transformações necessárias antes de predizer a emissão de CO2. Isso torna o sistema **pronto para produção** sem etapas manuais de pré-processamento.

---
_Próximo passo: `04_model_training.ipynb` — Treinamento, comparação e seleção do melhor modelo._